# Clase 11 — Traducción de voz: el proyecto Babel Fish

## Objetivos de aprendizaje

- Entender la arquitectura STT → LLM → TTS aplicada a traducción (proyecto Babel Fish de IBM).
- Detectar por qué el prompt "traductor" original del material fuente no traduce de verdad.
- Construir un mini-traductor de voz 100% gratuito con Hugging Face (`Helsinki-NLP/opus-mt-en-es`) y, opcionalmente, darle voz con `gTTS`.
- Comparar la vía "LLM generalista + prompt" con la vía "modelo especializado en traducción".

## Qué vamos a construir

Un traductor de inglés a español que recibe texto (en un proyecto real, vendría de Whisper transcribiendo audio), lo traduce con un modelo gratuito de Hugging Face, y opcionalmente lee la traducción en voz alta.

## Repaso teórico: la arquitectura Babel Fish

El proyecto **Babel Fish** de IBM (`BabelFish.pdf`, proyecto `babel-fish-project/translator-with-voice-and-watsonx/`) reutiliza la misma arquitectura del asistente de voz conversacional:

```
[Navegador] --graba audio, POST--> [Flask server.py]
  POST /speech-to-text   -> Watson STT   -> texto
  POST /process-message  -> watsonx (Mistral) -> traducción -> Watson TTS -> audio
```

Tres funciones en `worker.py`: `speech_to_text`, `watsonx_process_message` (el LLM) y `text_to_speech`.

**El hallazgo real**: en la implementación que circula del proyecto, `watsonx_process_message` usa este prompt genérico:

```python
def watsonx_process_message(user_message):
    prompt = f"""Respond to the query: ```{user_message}```"""
    response_text = model.generate_text(prompt=prompt)
    return response_text.strip()
```

Eso **no** traduce nada — es un prompt de "responde a esto", no de "traduce esto". El propio PDF documenta el prompt correcto (especificando idioma de origen y destino), pero la implementación real no lo usa. Es un bug real del material fuente, y la lección es: **prueba tus prompts, no asumas que hacen lo que su nombre promete.**

Abajo tienes ese código real de watsonx como referencia — no lo vamos a ejecutar (requiere credenciales de pago). En vez de eso, construimos la misma idea con una alternativa 100% gratuita.

### Bloque de referencia — código real de watsonx (NO ejecutable sin credenciales de IBM)

Así se hace con las APIs de pago que usa el material oficial de IBM. Aquí abajo usamos la alternativa gratuita de Hugging Face para que puedas seguir la clase sin coste.

```python
from ibm_watson_machine_learning.foundation_models import Model

# Prompt CORRECTO documentado en el PDF (pero no usado en la implementación real):
def watsonx_process_message(user_message):
    prompt = f"""
    Translate the following English sentence into Spanish.
    Reply ONLY with the translation, no explanations, no formatting, no extra text.
    English: {user_message}
    Spanish:
    """
    response_text = model.generate_text(prompt=prompt)
    return response_text.strip()

# Prompt REAL encontrado en el proyecto (bug: no traduce de verdad)
def watsonx_process_message_bug(user_message):
    prompt = f"""Respond to the query: ```{user_message}```"""
    response_text = model.generate_text(prompt=prompt)
    return response_text.strip()
```

In [ ]:
# Instalamos las librerias que vamos a usar (gratuitas, sin API key).
# transformers + sentencepiece: para el modelo de traduccion de Hugging Face.
# gTTS: para convertir la traduccion en audio (Google Text-to-Speech, gratuito).
!pip install -q transformers sentencepiece gTTS

## Paso 1 — Cargar un modelo de traducción real y gratuito

`Helsinki-NLP/opus-mt-en-es` es un modelo **especializado en traducción** inglés→español (no un LLM generalista al que hay que "convencer" con un prompt). Esa es la diferencia clave con el bug de Babel Fish: este modelo solo sabe traducir, así que no puede "olvidarse" de hacerlo.

In [ ]:
from transformers import pipeline

# Cargamos un pipeline de traduccion ya listo: descarga el modelo la primera vez
# y despues lo reutiliza. No hace falta ninguna clave de API.
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")

# Probamos con una frase de ejemplo
resultado = translator("Where is the nearest train station?")
print(resultado[0]["translation_text"])

## Paso 2 — Reproducir el bug de Babel Fish para verlo con tus propios ojos

Vamos a simular el prompt genérico del bug real (`"Respond to the query: ..."`) usando el mismo modelo de traducción como si fuera un LLM al que le "pedimos" cualquier cosa con un prompt de texto plano. Como `Helsinki-NLP/opus-mt-en-es` es un modelo *seq2seq* especializado, no acepta instrucciones en lenguaje natural como un LLM — así que para ilustrar el bug de verdad, comparamos conceptualmente las dos funciones de `watsonx_process_message` de la celda de referencia de arriba.

La comparación real que importa es esta tabla mental:

| Función | Prompt | ¿Traduce de verdad? |
|---|---|---|
| `watsonx_process_message` (PDF, correcta) | pide explícitamente traducir EN→ES | Sí |
| `watsonx_process_message_bug` (proyecto real) | "responde a esto" genérico | No necesariamente |
| `translator(...)` (Hugging Face, esta clase) | no hace falta prompt, el modelo solo traduce | Sí, por diseño |

In [ ]:
# Comprobamos el traductor con varias frases distintas, incluyendo preguntas,
# ordenes y frases largas, para verificar que SIEMPRE traduce (no solo a veces).
frases = [
    "Good morning, how can I help you?",
    "The flight to Madrid has been delayed by two hours.",
    "Please, translate this sentence carefully.",
]

for frase in frases:
    traduccion = translator(frase)[0]["translation_text"]
    print(f"EN: {frase}")
    print(f"ES: {traduccion}")
    print("---")

## Paso 3 — Darle voz a la traducción (opcional, con gTTS)

`gTTS` (Google Text-to-Speech) convierte texto en un archivo de audio, de forma gratuita. Es el equivalente sencillo del Watson TTS que usa Babel Fish en la arquitectura real.

In [ ]:
from gtts import gTTS
from IPython.display import Audio

texto_original = "The flight to Madrid has been delayed by two hours."
traduccion = translator(texto_original)[0]["translation_text"]
print("Traduccion:", traduccion)

# Generamos el audio en espanol a partir de la traduccion (no del texto original).
tts = gTTS(text=traduccion, lang="es")
tts.save("traduccion.mp3")

# Reproducimos el audio directamente en el notebook.
Audio("traduccion.mp3")

## Resumen de la clase

- Babel Fish reutiliza la arquitectura STT → LLM → TTS del asistente de voz, cambiando el trabajo del LLM a "traducir".
- El material fuente tiene un bug real: el prompt usado en la implementación no fuerza traducción, a pesar de llamarse "traductor".
- Un modelo especializado en traducción (`Helsinki-NLP/opus-mt-en-es`) traduce por diseño, sin depender de que un prompt esté bien escrito.
- `gTTS` te permite añadir voz a cualquier texto de forma gratuita, cerrando el ciclo texto → voz.

**Sigue con:**
- Capítulo del manual: [15_traduccion_babelfish.md](../manual/15_traduccion_babelfish.md)
- Notebook de práctica: [clase11_practica_traduccion_babelfish.ipynb](../notebooks_practica/clase11_practica_traduccion_babelfish.ipynb)